In [1]:
# Clone the repository
!git clone https://github.com/phanindra-max/Cross-Generator-Generalization-Project.git

# Move into the directory
%cd Cross-Generator-Generalization-Project/

# Install dependencies if you have a requirements.txt
!pip install uv
!uv pip install -r requirements.txt

fatal: destination path 'Cross-Generator-Generalization-Project' already exists and is not an empty directory.
/content/Cross-Generator-Generalization-Project
Using Python 3.12.13 environment at: /usr
Checked 18 packages in 99ms


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Adjust this path if you keep the project elsewhere on Drive.
DRIVE_BACKUP = Path("/content/drive/MyDrive/deepfake-cross-generator/backups")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

# Source-of-truth list: every dataset artifact we want persisted.
ARTIFACTS = [
    ("data/real",           "real.tar",           "FFHQ thumbnails"),
    ("data/faceswap",       "faceswap.tar",       "FF++ FaceSwap frames"),
    ("data/diffusion_sd15", "diffusion_sd15.tar", "Stable Diffusion outputs"),
    ("data/splits",         "splits.tar",         "split manifest CSVs"),
]
print(f"Backup target: {DRIVE_BACKUP}")

# Data Pipeline: Deepfake Cross-Generator Detection

This notebook downloads and prepares all datasets for the project:
1. **FFHQ** (real faces) — 2000 thumbnails from HuggingFace
2. **FaceForensics++ FaceSwap c23** — 2000 frames from Kaggle
3. **Stable Diffusion v1.5** — 1000 generated portraits

Then creates reproducible CSV split manifests.

**Runtime**: Use a GPU runtime (T4 or better) for Stable Diffusion generation.

## 0. Install Dependencies

In [2]:
!uv pip install -q -r requirements.txt

In [3]:
import os
import sys
from pathlib import Path

# If running from notebooks/ directory, add project root to path
PROJECT_ROOT = Path(".").resolve().parent
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

Working directory: /content/Cross-Generator-Generalization-Project


## 1. Download FFHQ Real Faces (HuggingFace)

Downloads 2000 128x128 face thumbnails from the FFHQ dataset hosted on HuggingFace.

#### *Restore data from Drive (if available)

In [9]:
import subprocess, time
from pathlib import Path

def _is_empty(p: Path) -> bool:
    return (not p.exists()) or (not any(p.iterdir()))

def restore_one(local_dir: str, archive_name: str, label: str) -> bool:
    target = Path(local_dir)
    archive = DRIVE_BACKUP / archive_name
    if not _is_empty(target):
        print(f"  skip {label:<26} {local_dir} already populated")
        return True
    if not archive.exists():
        print(f"  miss {label:<26} no archive in Drive (pipeline will generate)")
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    t = time.time()
    subprocess.run(["tar", "-xf", str(archive), "-C", str(target.parent)], check=True)
    n_files = sum(1 for _ in target.iterdir())
    print(f"  ok   {label:<26} restored {n_files} files in {time.time()-t:.1f}s")
    return True

print("Restoring artifacts from Drive...")
restored = {label: restore_one(d, a, label) for d, a, label in ARTIFACTS}

# Final sanity check
print("\nLocal data tree:")
!ls data/ 2>/dev/null && echo "---" && for d in real faceswap diffusion_sd15 splits; do printf "%-20s %s files\n" "$d" "$(ls data/$d 2>/dev/null | wc -l)"; done

Restored real.tar in 0.2s
Restored faceswap.tar in 0.2s
faceswap  real	splits
---
00000.png
00001.png
00002.png
---
00000_00.png
00000_01.png
00000_02.png


In [10]:
from src.data.prepare_ffhq import download_ffhq

num_real = download_ffhq(
    output_dir="data/real",
    num_images=2000,
    resolution=128,
)
print(f"\nTotal real images: {num_real}")

Already have 2000 images in data/real, skipping download.

Total real images: 2000


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Download FaceForensics++ FaceSwap c23 (Kaggle)

Downloads the FF++ c23 dataset from Kaggle and extracts 4 evenly-spaced frames from 500 FaceSwap videos.

**Note**: You need Kaggle credentials configured. In Colab, run:
```python
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
```
Or upload your `kaggle.json` to `~/.kaggle/kaggle.json`.

In [12]:
# Configure Kaggle credentials (uncomment the method that works for you)

# Method 1: Colab Secrets
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Method 2: Direct (replace with your credentials)
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_key"

# Method 3: Upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [14]:
from src.data.prepare_faceforensics import prepare_faceforensics

num_faceswap = prepare_faceforensics(
    output_dir="data/faceswap",
    num_videos=500,
    frames_per_video=4,
    resolution=128,
)
print(f"\nTotal FaceSwap frames: {num_faceswap}")

Already have 2000 FaceSwap frames in data/faceswap, skipping download and extraction.

Total FaceSwap frames: 2000


Backup files to Drive after step 1 and 2


In [15]:
# Backup files from drive
# from google.colab import drive
# drive.mount("/content/drive")

# from pathlib import Path
# DRIVE_BACKUP = Path("/content/drive/MyDrive/deepfake-cross-generator/backups")
# DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
# print(f"Backup target: {DRIVE_BACKUP}")

In [16]:
# import subprocess, time
# from pathlib import Path

# def backup_dir(local_dir: str, archive_name: str):
#     src = Path(local_dir)
#     if not src.exists() or not any(src.iterdir()):
#         print(f"Skip {local_dir}: empty or missing")
#         return
#     dest = DRIVE_BACKUP / archive_name
#     n_files = sum(1 for _ in src.glob("*"))
#     print(f"Archiving {n_files} files from {src} -> {dest} ...")
#     t = time.time()
#     # -C parents makes the archive contain only the leaf dir name (real/, faceswap/),
#     # so restore is path-agnostic.
#     subprocess.run(
#         ["tar", "-cf", str(dest), "-C", str(src.parent), src.name],
#         check=True,
#     )
#     size_mb = dest.stat().st_size / (1024 * 1024)
#     print(f"  {size_mb:.1f} MB in {time.time()-t:.1f}s")

# backup_dir("data/real", "real.tar")
# backup_dir("data/faceswap", "faceswap.tar")
# # Splits and diffusion outputs (run after step 3 if you want):
# # backup_dir("data/splits", "splits.tar")
# # backup_dir("data/diffusion_sd15", "diffusion_sd15.tar")
# print("\nDone. Files in Drive:")
# !ls -lh /content/drive/MyDrive/deepfake-cross-generator/backups/

## 3. Generate Stable Diffusion v1.5 Faces

Generates 1000 portrait images using 20 diverse prompts (50 images each).
Takes ~30 minutes on a T4 GPU.

In [17]:
from src.data.generate_diffusion import generate_diffusion_faces

num_diffusion = generate_diffusion_faces(
    output_dir="data/diffusion_sd15",
    num_images=1000,
    resolution=128,
    batch_size=4,
    seed=42,
)
print(f"\nTotal diffusion images: {num_diffusion}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading Stable Diffusion pipeline: runwayml/stable-diffusion-v1-5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Generating 1000 images (50 per prompt)...


Prompts: 100%|██████████| 20/20 [09:49<00:00, 29.49s/it]

Done. Generated 1000 images in data/diffusion_sd15

Total diffusion images: 1000


## 4. Create Split Manifests

Produces three CSV files with deterministic splits:
- `train.csv`: 1600 real + 1600 FaceSwap
- `test_indist.csv`: 400 real + 400 FaceSwap  
- `test_ood.csv`: 400 real + 1000 Stable Diffusion fakes

In [ ]:
from src.data.create_splits import create_splits

train_df, test_indist_df, test_ood_df = create_splits(
    real_dir="data/real",
    faceswap_dir="data/faceswap",
    diffusion_dir="data/diffusion_sd15",
    output_dir="data/splits",
    seed=42,
)

#### Save to Drive

In [ ]:
import subprocess, time
from pathlib import Path

def backup_one(local_dir: str, archive_name: str, label: str):
    src = Path(local_dir)
    if not src.exists() or not any(src.iterdir()):
        print(f"  skip {label}: {local_dir} is empty/missing")
        return
    n_files = sum(1 for _ in src.iterdir())
    final = DRIVE_BACKUP / archive_name
    tmp = final.with_suffix(final.suffix + ".tmp")
    t = time.time()
    subprocess.run(
        ["tar", "-cf", str(tmp), "-C", str(src.parent), src.name],
        check=True,
    )
    tmp.replace(final)
    size_mb = final.stat().st_size / (1024 * 1024)
    print(f"  ok   {label:<26} {n_files:>5} files  {size_mb:>7.1f} MB  {time.time()-t:>5.1f}s")

print("Saving artifacts to Drive...")
for local_dir, archive_name, label in ARTIFACTS:
    backup_one(local_dir, archive_name, label)

print("\nDrive contents:")
!ls -lh /content/drive/MyDrive/deepfake-cross-generator/backups/

## 5. Verification & Sample Grid

In [ ]:
import pandas as pd

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)

for split_name in ["train", "test_indist", "test_ood"]:
    df = pd.read_csv(f"data/splits/{split_name}.csv")
    print(f"\n{split_name}.csv: {len(df)} total samples")
    print(f"  By source: {df['source'].value_counts().to_dict()}")
    print(f"  By label:  {{0: 'real', 1: 'fake'}} -> {df['label'].value_counts().to_dict()}")

# Verify all files exist
all_dfs = pd.concat([train_df, test_indist_df, test_ood_df])
missing = [p for p in all_dfs["path"] if not Path(p).exists()]
if missing:
    print(f"\nWARNING: {len(missing)} files not found!")
    print(f"  First few: {missing[:5]}")
else:
    print(f"\nAll {len(all_dfs)} files verified to exist.")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
fig.suptitle("Sample Images by Source", fontsize=14)

sources = [("data/real", "Real (FFHQ)"), ("data/faceswap", "FaceSwap (FF++)"), ("data/diffusion_sd15", "Stable Diffusion")]

for row, (img_dir, title) in enumerate(sources):
    img_files = sorted(Path(img_dir).glob("*.png"))
    samples = random.sample(img_files, min(6, len(img_files)))

    for col, img_path in enumerate(samples):
        img = Image.open(img_path)
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(title, fontsize=11, rotation=0, labelpad=80, va="center")

plt.tight_layout()
plt.savefig("results/figures/data_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sample grid saved to results/figures/data_samples.png")

## 6. (Optional) Copy Splits to Google Drive

If you want to persist the split CSVs across Colab sessions:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r data/splits /content/drive/MyDrive/deepfake-cross-generator/data/
# print("Splits saved to Google Drive.")